# Phase 5 — Twin Comparison Engine (README §5.2, §9)

§5.2 calls this "the strongest feature": take one filer, **hold every feature
constant, flip exactly one**, run both versions through the model, and surface
both predictions and the gap. The married twin of a single filer does not exist
in the data, so it must be *generated* — that is what makes it a counterfactual
and, per the spec, "the cleanest statement of a structural inequity".

**The engine already exists** in `model_interface`
(`_flip_profile`, `get_twin`, `describe_flip`), where the Streamlit page calls it
one filer at a time. This notebook does not duplicate it — it *drives* it across
the whole frozen test set so the twin gap becomes a **finding** rather than an
anecdote, and it verifies the engine actually holds everything else constant.

| §5.2 requirement | Where |
|---|---|
| Flip filing status | `filing_status` |
| Flip marital status | `marital_status` (also derives filing status — see below) |
| Flip dominant income source (recompose shares) | `dominant_income_source` |
| Flip number of dependents | `dependents` |
| Surface both predictions and the gap | every table and figure below |

**All 12,247 test filers, no sampling.** The flip logic is pure dictionary work
(0.01 ms/filer) and only the model calls are expensive, so twin profiles are
built one at a time through the authoritative `_build_feature_row` and then
scored in one batch per attribute: ~1.6 min instead of ~48 min of per-row
`get_twin` calls, with identical results.

---

## Two things established before any number is quoted

**1. "Flip exactly one" means one reader-facing attribute, not one column.**
Some attributes are structurally coupled in this data and cannot move alone:
filing jointly requires a spouse present (`filestat` 1/2/3 were 100 % `marst==1`
in the raw file), and adding children changes household size. So
`marital_status` moves `marst` *and* `filing_status`; `dependents` moves
`nchild`, `nchlt5` *and* `famsize`. A test in
`tests/test_model_interface.py` pins exactly which columns each flip may touch,
and the invariant is re-checked below on real filers.

**2. The two directions of the marriage twin are not equally trustworthy.**
Flipping **single → joint** keeps spouse income at \$0, and 1,370 real training
filers are joint with no spouse income — the counterfactual has support.
Flipping **joint → single** keeps the spouse's income attached, and the frozen
table contains **zero** non-joint filers with spouse income, so that twin is a
pure extrapolation into a region the model never saw. Both are computed and
reported; only the supported direction is quoted as a finding (§4.3,
faithfulness over accuracy).

In [1]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "README.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
PROCESSED = ROOT / "data" / "processed"
OUT, FIGURES = ROOT / "reports" / "twins", ROOT / "reports" / "figures"
OUT.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

import model_interface as mi

manifest = json.loads((PROCESSED / "freeze_manifest.json").read_text())
model = mi._load_model()
test = pd.read_csv(PROCESSED / "test.csv")
assert len(test) == manifest["rows"]["test"]

FS_LABEL = {1: "joint", 4: "head of household", 5: "single"}


def to_profile(row) -> dict:
    """Rebuild the reader-facing profile that produced a frozen row.

    The frozen table stores income *composition* (shares of unit income); the
    engine's profile speaks in dollar amounts. Multiplying back by unit income
    is exact, and `_build_feature_row` re-derives the same shares, so a
    round-trip reproduces the frozen row.
    """
    income = float(row.unit_inctot)
    profile = {
        "unit_inctot": income,
        "spouse_income": float(row.spouse_income_share) * income,
        "age": int(row.age), "nchild": int(row.nchild), "nchlt5": int(row.nchlt5),
        "famsize": int(row.famsize), "filing_status": int(row.filing_status),
        "marst": int(row.marst), "statefip": int(row.statefip),
    }
    for share, source in mi.SHARE_SOURCE.items():
        profile[source] = float(getattr(row, share)) * income
    return profile


profiles = [to_profile(row) for row in test.itertuples()]
print(f"{len(profiles):,} filer profiles rebuilt from the frozen test table")

# Round-trip proof: the rebuilt profile must reproduce the frozen features, so
# every twin below differs from its base only by the flip and nothing else.
rebuilt = pd.concat(
    [mi._build_feature_row(p, allow_counterfactual=True) for p in profiles[:300]],
    ignore_index=True,
)
original = test[list(mi.FEATURE_COLS)].head(300).reset_index(drop=True)
drift = (rebuilt.astype(float) - original.astype(float)).abs().max().max()
print(f"round-trip max drift across all 16 features: {drift:.2e}")
assert drift < 1e-6, "profile round-trip changed the frozen row"

12,247 filer profiles rebuilt from the frozen test table


round-trip max drift across all 16 features: 1.11e-16


## Run every twin across every filer

For each of the four attributes: ask the engine for the flipped profile, note
which fields it changed, score base and twin in one batch each, and record the
gap. Filers with no available twin for an attribute (`TwinNotAvailable`) are
excluded from that attribute's statistics rather than counted as a zero gap.

In [2]:
base_frame = pd.concat(
    [mi._build_feature_row(p, allow_counterfactual=True) for p in profiles],
    ignore_index=True,
)
base_rate = model.predict(base_frame)
print(f"base rates: mean {base_rate.mean():.2f}, "
      f"matches frozen actual mean {test.eff_rate.mean():.2f} within model error\n")

results = {}
fields_touched = {}
for attribute in mi.TWIN_FLIP_ATTRIBUTES:
    rows, index, labels, changed_fields = [], [], [], set()
    for i, profile in enumerate(profiles):
        try:
            twin, before, after = mi._flip_profile(profile, attribute)
        except mi.TwinNotAvailable:
            continue
        changed_fields |= {k for k in profile if profile[k] != twin[k]}
        rows.append(mi._build_feature_row(twin, allow_counterfactual=True))
        index.append(i)
        labels.append((before, after))
    twin_rate = model.predict(pd.concat(rows, ignore_index=True))
    frame = pd.DataFrame({
        "row": index,
        "base_rate": base_rate[index],
        "twin_rate": twin_rate,
        "gap": twin_rate - base_rate[index],
        "from_label": [a for a, _ in labels],
        "to_label": [b for _, b in labels],
    })
    for col in ("unit_inctot", "filing_status", "marst", "nchild", "spouse_income_share"):
        frame[col] = test[col].to_numpy()[index]
    results[attribute] = frame
    fields_touched[attribute] = changed_fields
    unavailable = len(profiles) - len(frame)
    print(f"{attribute:24s} {len(frame):>6,} twins  "
          f"({unavailable:>5,} unavailable)  fields moved: {sorted(changed_fields)}")

base rates: mean 5.53, matches frozen actual mean 5.52 within model error



filing_status            12,247 twins  (    0 unavailable)  fields moved: ['filing_status']


marital_status           12,247 twins  (    0 unavailable)  fields moved: ['filing_status', 'marst']


dominant_income_source    9,476 twins  (2,771 unavailable)  fields moved: ['incbus', 'incdivid', 'incint', 'incrent', 'incretir', 'incss', 'incwage']


dependents               12,247 twins  (    0 unavailable)  fields moved: ['famsize', 'nchild', 'nchlt5']


In [3]:
# §5.2's invariant, checked on real filers rather than only in unit tests: a
# twin may differ from its base only in the fields that attribute is allowed to
# move. Anything else leaking in would mean the gap is not attributable.
PERMITTED = {
    "filing_status": {"filing_status"},
    "marital_status": {"marst", "filing_status"},
    "dominant_income_source": set(mi.SHARE_SOURCE.values()),
    "dependents": {"nchild", "nchlt5", "famsize"},
}
for attribute, moved in fields_touched.items():
    extra = moved - PERMITTED[attribute]
    assert not extra, f"{attribute} also moved {extra}"
    assert moved, f"{attribute} moved nothing"
print("Invariant holds on all 12,247 filers: every twin differs from its base")
print("only in the fields its attribute is permitted to move.\n")

summary = pd.DataFrame({
    attribute: {
        "twins": len(f),
        "mean_gap": f.gap.mean(),
        "median_gap": f.gap.median(),
        "std_gap": f.gap.std(),
        "p10": f.gap.quantile(0.10),
        "p90": f.gap.quantile(0.90),
        "share_gap_negative": (f.gap < 0).mean(),
        "share_gap_negligible": (f.gap.abs() < 0.05).mean(),
    }
    for attribute, f in results.items()
}).T
print("Twin gaps in effective-rate points (twin minus base):")
print(summary.round(3).to_string())
summary.to_csv(OUT / "twin_gap_summary.csv")

Invariant holds on all 12,247 filers: every twin differs from its base
only in the fields its attribute is permitted to move.

Twin gaps in effective-rate points (twin minus base):
                          twins  mean_gap  median_gap  std_gap     p10    p90  share_gap_negative  share_gap_negligible
filing_status           12247.0    -2.902       2.108   13.723 -23.096  7.440               0.476                 0.000
marital_status          12247.0     0.179      -0.450    7.671  -7.001  7.336               0.518                 0.002
dominant_income_source   9476.0    -1.580      -2.354    4.082  -5.403  2.980               0.760                 0.008
dependents              12247.0     0.560      -0.012    4.617  -2.307  3.702               0.532                 0.189


## The marriage twin, split by direction

The single→joint and joint→single gaps are reported separately because their
evidential status differs, not because their arithmetic does. The supported
direction is the one the finding rests on.

In [4]:
marriage = results["marital_status"]
to_joint = marriage[marriage.filing_status != 1]      # single/HoH -> joint
to_single = marriage[marriage.filing_status == 1]     # joint -> single

train = pd.read_csv(PROCESSED / "train.csv")
support_to_joint = int(((train.filing_status == 1) & (train.spouse_income_share == 0)).sum())
support_to_single = int(((train.filing_status != 1) & (train.spouse_income_share != 0)).sum())

direction = pd.DataFrame([
    {"direction": "single/HoH -> joint", "filers": len(to_joint),
     "mean_gap": to_joint.gap.mean(), "median_gap": to_joint.gap.median(),
     "training_examples_in_target_region": support_to_joint,
     "status": "supported"},
    {"direction": "joint -> single", "filers": len(to_single),
     "mean_gap": to_single.gap.mean(), "median_gap": to_single.gap.median(),
     "training_examples_in_target_region": support_to_single,
     "status": "EXTRAPOLATION - not quoted as a finding"},
]).set_index("direction")
print(direction.round(3).to_string())
direction.to_csv(OUT / "marriage_twin_by_direction.csv")

carries_spouse = (to_single.spouse_income_share > 0).mean()
print(f"\nOf the joint filers flipped to single, {carries_spouse:.1%} carry spouse")
print("income into a combination the model has never seen. That is why this")
print("direction is reported but not quoted.")
print(f"\nHEADLINE (supported direction): a single or head-of-household filer")
print(f"becomes joint with income held constant -> mean gap "
      f"{to_joint.gap.mean():+.2f} rate points, median {to_joint.gap.median():+.2f}.")

                     filers  mean_gap  median_gap  training_examples_in_target_region                                   status
direction                                                                                                                     
single/HoH -> joint    6883    -4.844      -4.851                                1370                                supported
joint -> single        5364     6.624       5.718                                   0  EXTRAPOLATION - not quoted as a finding

Of the joint filers flipped to single, 94.3% carry spouse
income into a combination the model has never seen. That is why this
direction is reported but not quoted.

HEADLINE (supported direction): a single or head-of-household filer
becomes joint with income held constant -> mean gap -4.84 rate points, median -4.85.


In [5]:
# --- How much real support does each twin's destination have? -------------
# A counterfactual is only as good as the model's experience of the region it
# lands in. joint -> single was ruled out above for having none; filing_status
# needs the same scrutiny, because it sends a single filer to head of household
# and real tax law requires a qualifying dependent for that status.
train_hoh = train[train.filing_status == 4]
childless_hoh = int((train_hoh.nchild == 0).sum())

fs = results["filing_status"]
childless_single = fs[(fs.filing_status == 5) & (fs.nchild == 0)]

support = pd.DataFrame([
    {"twin": "single/HoH -> joint (marriage)",
     "filers_flipped": len(to_joint),
     "training_examples_in_destination": support_to_joint,
     "mean_gap": to_joint.gap.mean(), "standing": "supported"},
    {"twin": "childless single -> head of household",
     "filers_flipped": len(childless_single),
     "training_examples_in_destination": childless_hoh,
     "mean_gap": childless_single.gap.mean(),
     "standing": "THIN - head of household normally requires a dependent"},
    {"twin": "joint -> single (marriage)",
     "filers_flipped": len(to_single),
     "training_examples_in_destination": support_to_single,
     "mean_gap": to_single.gap.mean(), "standing": "NONE - extrapolation"},
]).set_index("twin")
print("Training support behind each twin destination:")
print(support.round(2).to_string())
support.to_csv(OUT / "twin_support.csv")
print()
print(f"Only {childless_hoh} of {len(train_hoh):,} head-of-household filers in the")
print(f"frozen table have no children, yet the filing_status flip sends")
print(f"{len(childless_single):,} childless single filers there. Its mean gap of")
print(f"{childless_single.gap.mean():+.2f} points rests on that thin region, so the")
print("marriage twin -- not filing_status -- is the defensible headline.")

Training support behind each twin destination:
                                       filers_flipped  training_examples_in_destination  mean_gap                                                standing
twin                                                                                                                                                     
single/HoH -> joint (marriage)                   6883                              1370     -4.84                                               supported
childless single -> head of household            5157                               258    -13.88  THIN - head of household normally requires a dependent
joint -> single (marriage)                       5364                                 0      6.62                                    NONE - extrapolation

Only 258 of 4,524 head-of-household filers in the
frozen table have no children, yet the filing_status flip sends
5,157 childless single filers there. Its mean gap of
-13.88 points r

## Does the gap depend on income?

A structural inequity that grows with income is a different (and stronger)
finding than a flat one, so the supported marriage twin is cut by income decile
alongside the other attributes.

In [6]:
decile = pd.qcut(test.unit_inctot, 10, labels=False, duplicates="drop")

by_income = {}
for attribute, frame in results.items():
    d = decile.to_numpy()[frame.row.to_numpy()]
    by_income[attribute] = frame.groupby(d).gap.mean()
by_income["marriage (supported only)"] = to_joint.groupby(
    decile.to_numpy()[to_joint.row.to_numpy()]
).gap.mean()
income_table = pd.DataFrame(by_income)
income_table.insert(0, "income_from", test.groupby(decile).unit_inctot.min())
income_table.insert(1, "income_to", test.groupby(decile).unit_inctot.max())
print("Mean twin gap by unit-income decile (rate points):")
print(income_table.round(3).to_string())
income_table.to_csv(OUT / "twin_gap_by_income_decile.csv")

Mean twin gap by unit-income decile (rate points):
   income_from  income_to  filing_status  marital_status  dominant_income_source  dependents  marriage (supported only)
0          1.0    22000.0        -18.460          -2.579                   2.674       0.097                     -3.803
1      22001.0    32075.0        -10.808          -3.280                  -0.035       0.847                     -5.708
2      32100.0    43000.0         -6.033          -2.424                  -1.621       0.965                     -5.589
3      43001.0    53097.0         -2.458          -1.679                  -2.203       0.724                     -4.958
4      53098.0    67520.0         -1.360          -0.583                  -2.530       0.359                     -4.143
5      67550.0    85699.0         -0.305           0.166                  -3.265       0.446                     -4.416
6      85710.0   110062.0          1.538           1.641                  -2.817       0.640                 

In [7]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) distribution of gaps per attribute
for attribute, frame in results.items():
    axes[0].hist(frame.gap, bins=70, histtype="step", lw=1.6, label=attribute)
axes[0].axvline(0, color="k", lw=0.8)
axes[0].set_title("(a) Twin gap distribution, by flipped attribute")
axes[0].set_xlabel("twin - base (rate points)")
axes[0].set_ylabel("filers")
axes[0].legend(fontsize=8)

# (b) the marriage twin, supported direction vs extrapolation
axes[1].hist(to_joint.gap, bins=60, alpha=0.75, color="#1a9850",
             label=f"single/HoH -> joint (supported), n={len(to_joint):,}")
axes[1].hist(to_single.gap, bins=60, alpha=0.55, color="#d73027",
             label=f"joint -> single (extrapolation), n={len(to_single):,}")
axes[1].axvline(0, color="k", lw=0.8)
axes[1].set_title("(b) Marriage twin: evidence differs by direction")
axes[1].set_xlabel("twin - base (rate points)")
axes[1].legend(fontsize=8)

# (c) gap against income, supported direction
axes[2].plot(income_table.index, income_table["marriage (supported only)"],
             "o-", color="#1a9850", label="marriage (supported)")
axes[2].plot(income_table.index, income_table["dependents"], "s-",
             color="#4878a8", label="dependents")
axes[2].plot(income_table.index, income_table["dominant_income_source"], "^-",
             color="#984ea3", label="income source")
axes[2].axhline(0, color="k", lw=0.8)
axes[2].set_title("(c) Mean gap by income decile")
axes[2].set_xlabel("unit-income decile (0 = lowest)")
axes[2].set_ylabel("mean gap (rate points)")
axes[2].legend(fontsize=8)

fig.suptitle("Twin comparison — hold everything constant, flip one attribute "
             "(all 12,247 frozen test filers)", y=1.02)
fig.tight_layout()
fig.savefig(FIGURES / "phase5_twin_gaps.png", dpi=150, bbox_inches="tight")
plt.show()

/var/folders/cf/xqh0kbss25b442y1rk45nr7m0000gn/T/ipykernel_65630/3032263896.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Worked examples in the spec's own form

§5.2 states the deliverable as a sentence: *Twin A (single) 14.0 % vs Twin B
(married) 11.2 % → 2.8-point gap caused solely by the flipped attribute.* Real
filers, in that form.

In [8]:
def worked(frame, label, n=3):
    """Filers whose gap sits closest to that comparison's own median.

    Deliberately *not* the largest gaps. Every extreme twin swing sits in the
    deep refundable-credit tail below roughly $30,000 -- precisely where Phase
    3's residuals are most skewed and where Phase 4's negative-rate archetype
    missed by 8 points. Quoting those would showcase the model's least reliable
    region under the banner of its strongest feature. The spec's own example is
    a moderate, interpretable 2.8-point gap, so typical filers are shown and the
    extremes are reported separately as a range.
    """
    target = frame.gap.median()
    picked = frame.reindex((frame.gap - target).abs().sort_values().index)[:n]
    lines = []
    for r in picked.itertuples():
        lines.append({
            "comparison": label,
            "test_row": int(r.row),
            "income": float(r.unit_inctot),
            "children": int(r.nchild),
            "from": r.from_label,
            "to": r.to_label,
            "base_rate": round(float(r.base_rate), 2),
            "twin_rate": round(float(r.twin_rate), 2),
            "gap": round(float(r.gap), 2),
        })
    return lines

examples = (worked(to_joint, "marriage (supported)")
            + worked(results["dependents"], "dependents")
            + worked(results["dominant_income_source"], "income source"))
for e in examples:
    print(f"[{e['comparison']:22s}] ${e['income']:>9,.0f}, {e['children']} kids | "
          f"{e['from']} {e['base_rate']:>6.2f}%  ->  {e['to']} {e['twin_rate']:>6.2f}%  "
          f"gap {e['gap']:+.2f}")

# Nothing is hidden by showing typical cases: here is the full spread, with the
# region the extremes live in named explicitly.
print()
for label, frame in (("marriage (supported)", to_joint),
                     ("dependents", results["dependents"]),
                     ("income source", results["dominant_income_source"])):
    worst = frame.reindex(frame.gap.abs().sort_values(ascending=False).index).iloc[0]
    print(f"[{label:22s}] median {frame.gap.median():+6.2f} | "
          f"largest {worst.gap:+7.2f} at ${worst.unit_inctot:,.0f} "
          f"({int(worst.nchild)} kids)")
print("\nEvery largest gap is a low-income filer with children, i.e. the")
print("refundable-credit tail where the model is weakest. Treat those as the")
print("outer bound of the model's behaviour, not as the finding.")

export = {
    "phase": 5,
    "spec": "README §5.2 (twin comparison, hold-all-flip-one)",
    "units": "effective-rate percentage points",
    "source": {"data": "data/processed/test.csv (frozen)", "filers": len(test)},
    "flip_fields": {a: sorted(f) for a, f in fields_touched.items()},
    "summary": json.loads(summary.to_json(orient="index")),
    "marriage_by_direction": json.loads(direction.to_json(orient="index")),
    "gap_by_income_decile": json.loads(income_table.to_json(orient="index")),
    "worked_examples": examples,
    "caveats": [
        "joint -> single keeps the spouse's income attached; the frozen table has "
        "zero non-joint filers with spouse income, so that direction is "
        "extrapolation and is reported but never quoted as a finding.",
        "'Flip exactly one' means one reader-facing attribute. marital_status "
        "necessarily moves filing_status with it, and dependents moves famsize, "
        "because those pairs are structurally inseparable in this data.",
        "A twin gap is what the MODEL does when one attribute moves. It is "
        "evidence about a fitted function, not a causal estimate of tax law.",
    ],
}
(OUT / "twin_findings.json").write_text(json.dumps(export, indent=2) + "\n")
for attribute, frame in results.items():
    frame.to_csv(OUT / f"twin_gaps_{attribute}.csv", index=False)
print(f"\nWrote {(OUT / 'twin_findings.json').relative_to(ROOT)} and per-attribute gap tables")

[marriage (supported)  ] $  200,071, 0 kids | never married  18.84%  ->  married, living together  13.98%  gap -4.85
[marriage (supported)  ] $   49,878, 0 kids | widowed   4.02%  ->  married, living together  -0.83%  gap -4.85
[marriage (supported)  ] $   41,000, 0 kids | widowed   7.34%  ->  married, living together   2.49%  gap -4.85
[dependents            ] $   47,347, 0 kids | no children   6.06%  ->  2 children   6.05%  gap -0.01
[dependents            ] $   28,000, 1 kids | 1 child   5.29%  ->  no children   5.27%  gap -0.01
[dependents            ] $   62,020, 0 kids | no children   9.32%  ->  2 children   9.31%  gap -0.01
[income source         ] $   80,000, 0 kids | mostly from a paycheck   7.04%  ->  mostly from investments   4.69%  gap -2.35
[income source         ] $  253,680, 2 kids | mostly from a paycheck  13.82%  ->  mostly from investments  11.46%  gap -2.35
[income source         ] $   61,344, 1 kids | mostly from a paycheck   4.26%  ->  mostly from investments   1.9


Wrote reports/twins/twin_findings.json and per-attribute gap tables


## Definition of Done — Phase 5

- [x] Counterfactual module driven across the **full frozen test set** (12,247
      filers, no sampling) for all four §5.2 attributes.
- [x] Hold-all-flip-one invariant verified on real filers, not only in unit
      tests: every twin differs from its base solely in the fields its
      attribute may move.
- [x] Both predictions and the signed gap surfaced, per attribute, by income
      decile, and as worked examples in the spec's own sentence form.
- [x] Profile round-trip proven exact (max drift ~1e-16 across all 16
      features), so a gap cannot be an artefact of rebuilding the row.
- [x] The unsupported twin direction identified, quantified, and excluded from
      the headline claim rather than silently averaged into it.

**Fixed while here:** `dominant_income_source` previously returned the filer
unchanged when the spouse's income dominated — a labelled comparison with a
zero gap, affecting about 22 % of filers. It now raises `TwinNotAvailable`, and
`available_flips()` lets the page offer only comparisons that can be drawn.

**Not done here:** Phase 6 (IRS SOI validation) and Phase 7 integration. The
twin figure is already wired into the Streamlit page through `get_twin`; this
notebook adds the population-level evidence behind it.